In [1]:
import os
import json
import gzip
import pandas as pd

from dotenv import load_dotenv

from typing import TypedDict

from langchain_core.documents import Document

from langchain_core.tools import Tool

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_huggingface import (
    HuggingFaceEmbeddings
)

from langchain_community.vectorstores import FAISS

from langchain_community.retrievers import (
    BM25Retriever
)

from langchain_classic.retrievers import EnsembleRetriever

from langchain_groq import ChatGroq

from langchain_community.tools import (
    DuckDuckGoSearchRun
)

from langgraph.graph import (
    StateGraph,
    END
)

d:\commerce_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [3]:
faq_docs = []

with open(
    "D:/commerce_chatbot/data/faq/train.json",
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        item = json.loads(line)

        faq_docs.append(
            Document(
                page_content=f"""
                Question: {item['question']}
                Answer: {item['answer']}
                """,
                metadata={
                    "source": "faq"
                }
            )
        )

print("FAQ Docs:", len(faq_docs))

FAQ Docs: 79


In [4]:
print(faq_docs[0])

page_content='
                Question: How can I create an account?
                Answer: To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.
                ' metadata={'source': 'faq'}


In [5]:
from langchain_community.document_loaders import (
    PyPDFLoader
)

loader = PyPDFLoader(
    "D:/commerce_chatbot/data/pdfs/employee_handbook_print_1.pdf"
)

pdf_docs = loader.load()

print("PDF Docs:", len(pdf_docs))

PDF Docs: 90


In [6]:
products_df = pd.read_csv(
    "D:/commerce_chatbot/data/csv/amazon-products.csv"
)

# Reduce dataset size for faster embeddings
products_df = products_df.head(1000)

products_docs = []

for _, row in products_df.iterrows():

    products_docs.append(
        Document(
            page_content=str(row.to_dict()),
            metadata={
                "source": "products"
            }
        )
    )

print("Product Docs:", len(products_docs))

Product Docs: 1000


In [7]:
print(products_docs[0].page_content)

{'Unnamed: 0': 0, 'name': 'Lloyd 1.5 Ton 3 Star Inverter Split Ac (5 In 1 Convertible, Copper, Anti-Viral + Pm 2.5 Filter, 2023 Model, White, Gls18I3...', 'main_category': 'appliances', 'sub_category': 'Air Conditioners', 'image': 'https://m.media-amazon.com/images/I/31UISB90sYL._AC_UL320_.jpg', 'link': 'https://www.amazon.in/Lloyd-Inverter-Convertible-Anti-Viral-GLS18I3FWAMC/dp/B0BRKXTSBT/ref=sr_1_4?qid=1679134237&s=kitchen&sr=1-4', 'ratings': '4.2', 'no_of_ratings': '2,255', 'discount_price': '₹32,999', 'actual_price': '₹58,990'}


In [8]:
support_df = pd.read_csv(
    "D:/commerce_chatbot/data/csv/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"
)

# Reduce dataset size for faster embeddings
support_df = support_df.head(2000)

support_docs = []

for _, row in support_df.iterrows():

    support_docs.append(
        Document(
            page_content=str(row.to_dict()),
            metadata={
                "source": "support"
            }
        )
    )

print("Support Docs:", len(support_docs))

Support Docs: 2000


In [9]:
print(support_docs[0].page_content)

{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}


In [10]:
qa_docs = []

with gzip.open(
    "D:/commerce_chatbot/data/qa/amazon-qa.jsonl.gz",
    "rt",
    encoding="utf-8"
) as f:

    for i, line in enumerate(f):

        # Reduce dataset size for faster embeddings
        if i >= 2000:
            break

        item = json.loads(line)

        qa_docs.append(
            Document(
                page_content=str(item),
                metadata={
                    "source": "amazon_qa"
                }
            )
        )

print("Amazon QA Docs:", len(qa_docs))

Amazon QA Docs: 2000


In [11]:
print(qa_docs[0].page_content)

{'query': 'does this fit the z2x version? Thx', 'pos': ['I am not 100% sure. It appears that it does based on the size of the torch housing. I have the 9p and it fits perfectly. Compare the two and see. Otherwise it is a great product and has lasted 2 years so far wearing every day on my duty belt. Gets lots of inquiries and compliments.', 'Yes. The retention clip on this holster holds the bezel, the shape of the flashlight shaft is irrelevant. I would suggest getting a different holster however, since the retention clip is plastic and does not work very well.']}


In [12]:
print("FAQ:", len(faq_docs))

print("PDF:", len(pdf_docs))

print("Products:", len(products_docs))

print("Support:", len(support_docs))

print("Amazon QA:", len(qa_docs))

FAQ: 79
PDF: 90
Products: 1000
Support: 2000
Amazon QA: 2000


In [13]:
all_docs = (
    faq_docs
    + pdf_docs
    + products_docs
    + support_docs
    + qa_docs
)

print(len(all_docs))

5169


In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

split_docs = splitter.split_documents(
    all_docs
)

print("Chunks:", len(split_docs))

Chunks: 7663


In [15]:
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2166.55it/s]


In [16]:
from tqdm import tqdm

batch_size = 100

for i in tqdm(
    range(0, len(split_docs), batch_size)
):

    batch = split_docs[i:i+batch_size]

    if i == 0:

        db = FAISS.from_documents(
            batch,
            embedding
        )

    else:

        db.add_documents(batch)

db.save_local("vectordb")

100%|██████████| 77/77 [08:41<00:00,  6.77s/it]


In [27]:
results = db.similarity_search(
    "refund policy",
    k=3
)

for r in results:
    print(r.page_content)
    print("="*50)

Question: What is your return policy?
                Answer: Our return policy allows you to return products within 30 days of purchase for a full refund, provided they are in their original condition and packaging. Please refer to our Returns page for detailed instructions.
based on your specific situation.\n \n2. Check for Cancellation Policies: Review our cancellation policies, terms, and conditions, as they might offer insights into your available options. Look for sections like "{{Cancellation Policy}}" or "{{Refund Policy}}" on our website FAQs or Help Center.\n\n3. Explore Refund or Return Options: If cancelling your purchase is not feasible, you might consider exploring our refund or return policies. Look for sections such as "{{Refund Policy}}" or "{{Return Policy}}" on our website for detailed instructions on how to proceed.\n\n4. Explore Flexible Payment Options: We understand that financial circumstances may change. We offer flexible payment options to help our customers d

In [18]:
vector_retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.7
    }
)

docs = vector_retriever.invoke(
    "damaged product return"
)

for d in docs:
    print(d.page_content)
    print("="*50)

Question: Can I return a product if it was damaged due to improper use?
                Answer: Our return policy generally covers products that are defective or damaged upon arrival. Damage due to improper use may not be eligible for a return. Please contact our customer support team for assistance.
{'query': 'how can i return for refund', 'pos': ["Hey Paula; Don't expect to be able to return it, due to laws about gas/fuel items and returning. Contact Amazon. They are very good about solving the problem, even to the point of sending another complete unit, or working out a refund..."]}
damage being occurred. Add to that that the UPS box stays sitting in the heat outside for hours before you can retrieve the box...and you have a problem.']}
Question: Can I return a product if it was damaged during shipping?
                Answer: If your product was damaged during shipping, please contact our customer support team immediately. We will guide you through the return and replacement proces

In [19]:
bm25_retriever = BM25Retriever.from_documents(
    split_docs
)

bm25_retriever.k = 5

docs = bm25_retriever.invoke(
    "refund policy"
)

for d in docs:
    print(d.page_content)

{'flags': 'BLN', 'instruction': "I can't afford purchase {{Order Number}}", 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I'm on your side your concern about being unable to afford the purchase with the purchase number {{Order Number}}. Our main priority is to assist you in resolving this matter and ensuring your satisfaction. Let's explore some possible options together:\n\n1. Review Refund Policies: Our company has specific refund policies in place to cater to situations like this. I recommend reviewing our refund policy page on our website to understand the options available to you.\n\n2. Contact Customer Support: Reach out to our dedicated customer support team through {{Customer Support Hours}} at {{Customer Support Phone Number}}. They are equipped with the knowledge
{'flags': 'BELNQ', 'instruction': 'i cannot afford purchase {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I grasp that you are facing financial constraints and cannot af

In [20]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        bm25_retriever,
        vector_retriever
    ],
    weights=[0.4, 0.6]
)

docs = hybrid_retriever.invoke(
    "How do I return damaged products?"
)

for d in docs:
    print(d.page_content)
    print("="*50)

Question: Can I return a product if it was damaged due to improper use?
                Answer: Our return policy generally covers products that are defective or damaged upon arrival. Damage due to improper use may not be eligible for a return. Please contact our customer support team for assistance.
Question: Can I return a product if it was damaged during shipping?
                Answer: If your product was damaged during shipping, please contact our customer support team immediately. We will guide you through the return and replacement process.
{'flags': 'BK', 'instruction': 'remove item from order', 'category': 'ORDER', 'intent': 'change_order', 'response': "I've deduced from your message that you need assistance in removing an item from your order. Could you please provide me with the order number and the specific item you would like to remove? This information will help us expedite the process and ensure that your order is exactly as you want it. Thank you for bringing this to o

In [21]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    streaming=True
)

In [22]:
from langchain_core.prompts import (
    ChatPromptTemplate
)

from langchain_core.output_parsers import (
    StrOutputParser
)

prompt = ChatPromptTemplate.from_template("""
Answer the question using context.

Context:
{context}

Question:
{question}
""")

parser = StrOutputParser()

chain = prompt | llm | parser

In [23]:
context = "\n".join(
    [d.page_content for d in docs]
)

response = chain.invoke({
    "context": context,
    "question": "Can I return damaged products?"
})

print(response)

Our return policy generally covers products that are defective or damaged upon arrival. If your product was damaged during shipping, please contact our customer support team immediately and we will guide you through the return and replacement process. However, damage due to improper use may not be eligible for a return. Please contact our customer support team for assistance with your specific situation.


In [24]:
def rag_search(query):

    docs = hybrid_retriever.invoke(query)

    return "\n\n".join(
        [d.page_content for d in docs]
    )

rag_tool = Tool(
    name="Commerce KB",
    func=rag_search,
    description="Useful for ecommerce support"
)

In [25]:
from langchain_community.tools import (
    DuckDuckGoSearchRun
)

search = DuckDuckGoSearchRun()

In [26]:
duck_tool = Tool(
    name="DuckDuckGo Search",
    func=search.run,
    description="Useful for latest web info"
)

In [29]:
def web_search(query):

    result = duck_tool.invoke(query)

    return result

In [30]:
result = duck_tool.invoke(
    "latest Amazon Prime shipping policy"
)

print(result)

Starting Oct. 1, Amazon Prime customers wanting to share shipping benefits and added digital perks can do so using the Amazon Family program. Starting Oct. 1, Amazon will change its program that allows Prime members to share free shipping benefits with people outside their household. Shoppers Are FURIOUS About Amazon's New Shipping Policy—Here's What to Know Before Prime Day "The only way to get even is to start canceling our accounts," one angry shopper wrote. One of the biggest perks of an Amazon Prime membership is the fast, free shipping—and for a while, members were able to share this benefit. However, the retail giant has just announced that it's making changes to that policy. Amazon is eliminating a program that allows members of its Prime subscription program to share free shipping benefits with people outside their household.


In [31]:
def router(state):

    question = state["question"]

    if "latest" in question.lower():

        return "web"

    return "rag"

In [32]:
def rag_node(state):

    result = rag_search(
        state["question"]
    )

    return {
        "context": result
    }

In [33]:
def web_node(state):

    result = web_search(
        state["question"]
    )

    return {
        "context": result
    }

In [40]:
def generate(state):

    history = state.get(
        "history",
        []
    )

    prompt = f"""
    You are an AI commerce support agent.

    Previous Conversation:
    {history}

    Context:
    {state['context']}

    Question:
    {state['question']}
    """

    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

In [39]:
class GraphState(TypedDict):

    question: str

    context: str

    answer: str

    history: list

In [41]:
builder = StateGraph(GraphState)

builder.add_node(
    "rag",
    rag_node
)

builder.add_node(
    "web",
    web_node
)

builder.add_node(
    "generate",
    generate
)

builder.set_conditional_entry_point(
    router,
    {
        "rag": "rag",
        "web": "web"
    }
)

builder.add_edge(
    "rag",
    "generate"
)

builder.add_edge(
    "web",
    "generate"
)

builder.add_edge(
    "generate",
    END
)

graph = builder.compile()

In [42]:
history = []

In [43]:
response1 = graph.invoke({

    "question":
    "What is the refund policy?",

    "history":
    history
})

print(response1["answer"])

Our return policy allows you to return products within 30 days of purchase for a full refund, provided they are in their original condition and packaging. Please refer to our Returns page for detailed instructions.


In [44]:
history.append({

    "user":
    "What is the refund policy?",

    "assistant":
    response1["answer"]
})

In [45]:
response2 = graph.invoke({

    "question":
    "What about damaged products?",

    "history":
    history
})

print(response2["answer"])

If you receive a damaged product, please contact our customer support team immediately. We will assist you with the necessary steps for return, repair, or replacement, depending on the situation. If the damage occurred during shipping, we will guide you through the process of resolving the issue. Please have your order number and a description of the damage ready when you reach out to us, so we can help you as quickly as possible.


In [46]:
def stream_response(question):

    response = graph.invoke({
        "question": question,
        "history": []
    })

    return response["answer"]

In [47]:
print(
    stream_response(
        "How do I track my order?"
    )
)

You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment.


In [48]:
db.save_local("vectordb")

print("Vector DB Saved")

Vector DB Saved


In [49]:
db2 = FAISS.load_local(
    "vectordb",
    embedding,
    allow_dangerous_deserialization=True
)

print("Loaded Successfully")

Loaded Successfully


In [50]:
results = db2.similarity_search(
    "refund policy",
    k=3
)

for r in results:

    print(r.page_content)

    print("="*50)

Question: What is your return policy?
                Answer: Our return policy allows you to return products within 30 days of purchase for a full refund, provided they are in their original condition and packaging. Please refer to our Returns page for detailed instructions.
based on your specific situation.\n \n2. Check for Cancellation Policies: Review our cancellation policies, terms, and conditions, as they might offer insights into your available options. Look for sections like "{{Cancellation Policy}}" or "{{Refund Policy}}" on our website FAQs or Help Center.\n\n3. Explore Refund or Return Options: If cancelling your purchase is not feasible, you might consider exploring our refund or return policies. Look for sections such as "{{Refund Policy}}" or "{{Return Policy}}" on our website for detailed instructions on how to proceed.\n\n4. Explore Flexible Payment Options: We understand that financial circumstances may change. We offer flexible payment options to help our customers d